In [7]:
import pandas as pd
import numpy as np

pod_long = pd.read_excel("../data/raw/podlesny2022_abundance.xlsx")

pod_meta = pd.read_csv("../data/processed/podlesny_meta_split.csv")

pod_wide = pod_long.pivot_table(
    index='Name',
    columns='species',
    values='rel_abund',
    fill_value=0
)

pod_wide.to_pickle("../data/processed/podlesny_wide.pkl")

print("Podlesny wide:", pod_wide.shape)

records = []

for case in pod_meta['Case_Name'].dropna().unique():

    case_rows = pod_meta[pod_meta['Case_Name'] == case]

    donor_name = case_rows['Donor.Name'].iloc[0]
    split = case_rows['split'].iloc[0]
    disease = case_rows['Disease_Group'].iloc[0]

    donor_rows = pod_meta[pod_meta['Name'] == donor_name]
    if donor_rows.empty:
        continue

    donor_id = donor_rows['Name'].iloc[0]

    post_rows = case_rows[case_rows['Sample_Type'] == 'Post']

    late_post = post_rows[post_rows['Days_Since_FMT'] >= 28]

    post_rows_use = late_post if len(late_post) > 0 else post_rows

    if post_rows_use.empty:
        continue

    post_rows_use = post_rows_use.sort_values('Days_Since_FMT')

    post_id = post_rows_use['Name'].iloc[0]
    days = post_rows_use['Days_Since_FMT'].iloc[0]

    if donor_id not in pod_wide.index:
        continue
    if post_id not in pod_wide.index:
        continue

    donor_prof = pod_wide.loc[donor_id]
    post_prof = pod_wide.loc[post_id]

    for species in pod_wide.columns:

        if donor_prof[species] > 0.001:

            records.append({
                'case': case,
                'donor_id': donor_id,
                'post_id': post_id,
                'species': species,
                'donor_abundance': donor_prof[species],
                'post_abundance': post_prof[species],
                'engrafted': 1 if post_prof[species] > 0.001 else 0,
                'disease': disease,
                'days_post_fmt': days,
                'split': split
            })

labels_df = pd.DataFrame(records)

labels_df.to_csv(
    "../data/processed/engraftment_labels.csv",
    index=False
)

print("\nTotal records:", len(labels_df))
print("Engraftment rate:", round(labels_df['engrafted'].mean(), 3))
print("Cases covered:", labels_df['case'].nunique())

print("Train records:", (labels_df['split'] == 'train').sum())
print("Test records:", (labels_df['split'] == 'test').sum())

print("\nBy disease:")
print(
    labels_df.groupby('disease')['engrafted']
    .agg(['mean', 'count'])
    .round(3)
)

print("\n Engraftment labels saved")

Podlesny wide: (1419, 860)

Total records: 10951
Engraftment rate: 0.523
Cases covered: 203
Train records: 8807
Test records: 2144

By disease:
          mean  count
disease              
IBD      0.605   1483
ICI      0.596    969
MDR      0.501   1326
MetS     0.489   5606
rCDI     0.543   1567

 Engraftment labels saved


In [5]:
labels_df.head()
labels_df['engrafted'].value_counts(normalize=True)
labels_df['case'].nunique()

203

In [6]:
labels_df[['donor_abundance', 'post_abundance']].describe()

,donor_abundance,post_abundance
count,10951.000000,10951.000000
mean,0.018237,0.013643
std,0.038279,0.037129
min,0.001000,0.000000
25%,0.002243,0.000000
50%,0.005426,0.001268
75%,0.017235,0.009239
max,0.712712,0.563082
